# 🏋️‍♂️ Multi-Agent RAG with Microsoft Agent Framework Sequential Orchestration 🥑

Welcome to this fun, self-guided workshop where we build a multi-agent Retrieval-Augmented Generation (RAG) pipeline using **real Microsoft Agent Framework (MAF) Sequential Orchestration**. Our team of agents collaborates, one feeding the next, to answer fitness and health questions in an engaging way.

> **What changed from the previous version:** this notebook previously created two separate Foundry Prompt Agents by hand and manually piped the first agent's output into the second agent's input with custom Python glue code. That pattern doesn't scale, and it isn't how multi-agent chaining is meant to be expressed. This version uses MAF's `SequentialBuilder`, which is purpose-built for exactly this: a pipeline where each agent automatically receives the previous agent's output and adds its own response — no manual wiring required.

## 1. Setup

You'll import the required libraries and create a `FoundryChatClient` — the MAF-native chat client that talks to your Foundry project's model deployment. Authentication uses Microsoft Entra ID (keyless) via `AzureCliCredential`, so ensure `PROJECT_ENDPOINT` and `MODEL_DEPLOYMENT_NAME` are set in your `.env` and you are signed in (`az login`).

> **Note:** Agent Framework does not automatically load `.env` files — we call `load_dotenv()` explicitly below.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

# Load environment variables
notebook_path = Path().absolute()
env_path = notebook_path.parent.parent / '.env'  # Adjust path as needed
load_dotenv(env_path)

project_endpoint = os.environ["PROJECT_ENDPOINT"]
model_name = os.environ["MODEL_DEPLOYMENT_NAME"]

# FoundryChatClient is the Microsoft Agent Framework-native chat client for
# Foundry model deployments. This is a different code path from AIProjectClient
# + project_client.agents.create_version(...) used by Foundry Agent Service.
chat_client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model_name,
    credential=AzureCliCredential(),
)

print("✅ Foundry-backed Agent Framework chat client created successfully!")

## 2. Create Sample Health Data and Retrieval Tool

We'll define a small list of health tips and a simple retrieval function. This function simulates retrieving relevant health tips based on keywords in the user's query. We run this ourselves before the workflow starts, then hand the retrieved context to the pipeline as its input.

In [ ]:
# Define sample health tips
health_tips = [
    {"id": "tip1", "content": "Do a 10-minute HIIT workout to boost your metabolism.", "source": "Fitness Guru"},
    {"id": "tip2", "content": "Take a brisk 15-minute walk to clear your mind and improve circulation.", "source": "Health Coach"},
    {"id": "tip3", "content": "Stretch for 5 minutes every hour if you're sitting at a desk.", "source": "Wellness Expert"},
    {"id": "tip4", "content": "Incorporate strength training twice a week for overall fitness.", "source": "Personal Trainer"},
    {"id": "tip5", "content": "Drink water regularly to stay hydrated during workouts.", "source": "Nutritionist"}
]

def retrieve_tips(query: str) -> str:
    """Return health tips whose content contains keywords from the query."""
    query_lower = query.lower()
    relevant = []
    for tip in health_tips:
        if any(word in tip["content"].lower() for word in query_lower.split()):
            relevant.append(f"Source: {tip['source']} => {tip['content']}")
    if not relevant:
        relevant = [f"Source: {tip['source']} => {tip['content']}" for tip in health_tips]
    return "\n".join(relevant)

print("✅ Sample health tips and retrieval tool created!")

## 3. Define the Multi-Agent RAG Pipeline with Sequential Orchestration

We create two agents from the same `FoundryChatClient` using `.as_agent(...)`:

- **RetrieverAgent**: organizes the retrieved fitness and health tips for the query into a concise summary.
- **ResponderAgent**: crafts a fun, engaging final answer using RetrieverAgent's summary.

We then build a `SequentialBuilder` pipeline from these two agents. In sequential orchestration, each agent automatically receives the previous agent's output as part of the conversation — we no longer manually pass strings between agents ourselves.

In [ ]:
from agent_framework.orchestrations import SequentialBuilder

retriever_agent = chat_client.as_agent(
    name="RetrieverAgent",
    instructions=(
        "You are RetrieverAgent. You receive a user question along with retrieved tip context "
        "in the same message. Return a concise, useful summary of only the relevant tips."
    ),
)

responder_agent = chat_client.as_agent(
    name="ResponderAgent",
    instructions=(
        "You are ResponderAgent, a friendly fitness coach. Use the previous message's tip summary "
        "to craft an engaging answer to the user's original question. Include practical suggestions "
        "and a brief health disclaimer."
    ),
)

# Build sequential workflow: RetrieverAgent -> ResponderAgent
workflow = SequentialBuilder(participants=[retriever_agent, responder_agent]).build()

print("✅ Multi-agent RAG sequential pipeline defined!")

## 4. Test the Multi-Agent RAG System

Let's test our multi-agent RAG system with a fun fitness query. For example:

> **User Query:** I'm very busy but want to stay fit. What quick exercises can I do?

We retrieve the relevant tips ourselves first, then hand the combined question + context to the workflow as a single input. `SequentialBuilder` takes care of running RetrieverAgent first and passing its output to ResponderAgent — we just run the workflow once and read the final result.

In [ ]:
from agent_framework import AgentResponse

user_query = "I'm very busy but want to stay fit. What quick exercises can I do?"
retrieved_context = retrieve_tips(user_query)
print(f"User: {user_query}\n")

workflow_input = (
    f"User query:\n{user_query}\n\n"
    f"Retrieved tips:\n{retrieved_context}\n\n"
    "Summarize the most relevant tips, then craft a final engaging answer for the user."
)
events = await workflow.run(workflow_input)
outputs = events.get_outputs()

if outputs:
    final: AgentResponse = outputs[0]
    for msg in final.messages:
        name = msg.author_name or "assistant"
        print(f"[{name}]\n{msg.text}\n")
else:
    print("No output produced by the workflow.")

### Seeing Both Agents' Work (Intermediate Outputs)

By default, `SequentialBuilder` only surfaces the **last** agent's output (ResponderAgent above). If you want to see RetrieverAgent's intermediate summary too — useful for debugging or for showing your work — pass `intermediate_output_from` when building the workflow:

```python
workflow = SequentialBuilder(
    participants=[retriever_agent, responder_agent],
    intermediate_output_from=[retriever_agent],
).build()
```

Then, in streaming mode, both `"intermediate"` (RetrieverAgent) and `"output"` (ResponderAgent) events appear in the event stream, letting you display each agent's contribution as the pipeline runs.

## 5. Cleanup

Unlike Foundry Prompt Agents (created via `project_client.agents.create_version(...)`), agents created with `chat_client.as_agent(...)` are ephemeral and in-process — nothing was persisted on the server, so there's no agent version to delete here.

## 🎉 Conclusion

In this notebook, we built a fun multi-agent Retrieval-Augmented Generation pipeline using **Microsoft Agent Framework's Sequential Orchestration** with a fitness and health theme. We created a `RetrieverAgent` that organizes relevant health tips and a `ResponderAgent` that crafts an engaging answer to the user's query — chained together with `SequentialBuilder` instead of hand-written agent-to-agent glue code.

Feel free to modify and expand this notebook — for example, add a third agent (a "SafetyReviewerAgent" that checks the final answer for appropriate disclaimers) simply by adding it to the `participants` list. Happy coding and stay fit! 💪🥦